In [0]:
# 1. Get the list of unique years from your SQL table as a list of strings
year_data = spark.sql("SELECT DISTINCT YEAR(order_in_date) as yr FROM my_project_db.gold_sales_fact ORDER BY yr DESC").collect()
years = [str(row['yr']) for row in year_data]

# 2. Create the dropdown using the dynamic list
dbutils.widgets.dropdown("selected_year", years[0], years)

# 3. Retrieve the selected value for use in subsequent SQL cells
selected_year = dbutils.widgets.get("selected_year")

In [0]:
%sql
WITH SourceConversions AS (
    SELECT 
        p.division,
        l.source_name,
        COUNT(f.order_id) as total_conversions
    FROM my_project_db.gold_sales_fact f
    JOIN my_project_db.gold_dim_products p ON f.invt_id = p.invt_id
    JOIN my_project_db.gold_dim_leads l ON f.lead_id = l.lead_id
    -- This filter ensures the logic applies only to the selected year
    WHERE YEAR(f.order_in_date) = :selected_year
    GROUP BY p.division, l.source_name
),
RankedSources AS (
    SELECT 
        division,
        source_name,
        total_conversions,
        ROW_NUMBER() OVER(PARTITION BY division ORDER BY total_conversions DESC) as rank
    FROM SourceConversions
)
SELECT 
    division,
    source_name,
    total_conversions
FROM RankedSources 
WHERE rank <= 3
ORDER BY division, total_conversions DESC;